# Reproducing the EEG + NIRS competition experiments

Run this notebook from the package root after installing `requirements.txt`. The public dataset has 56 sessions. The judge streams one session at a time through the frozen runner. The cache for N17/N18 excludes both people; the final cache is separately trained on public data for hidden sessions.


In [ ]:
from pathlib import Path
import json, csv, subprocess, sys
ROOT = Path.cwd()
DATA = ROOT / "data_all"
print("Package:", ROOT)


## Obtain and verify the public data

The download script reads the public bucket index and checks each file length. The SHA256 manifest verifies every complete file. The test files are never included in the historical cache used to test them.


In [ ]:
subprocess.run([sys.executable, "download_sessions.py", "--out", str(DATA)], check=True)
subprocess.run([sys.executable, "verify_public_data.py", "--data", str(DATA)], check=True)


## Held-out subject evaluation

The supplied `results/holdout/artifacts_N17_N18` contains 46 usable sessions from other people. This cell replays all eight N17/N18 sessions and writes complete predictions, timings, and per-session metrics.


In [ ]:
subprocess.run([sys.executable, "strict_test.py", "--data", str(DATA), "--out", "results/notebook_strict_test"], check=True)


## Review the competition metrics

The score combines mean worst-class F1, mean macro recall, a NIRS bonus, and time penalties. Accuracy is diagnostic. The official aggregator averages session metrics; it does not merge confusion matrices.


In [ ]:
p = ROOT / "results/notebook_strict_test/metrics.json"
metric = json.loads(p.read_text(encoding="utf-8"))
print("mean min-F1", metric["hybrid"]["min_f1"])
print("macro recall", metric["hybrid"]["macro_recall"])
print("NIRS min-F1", metric["nirs_only"]["min_f1"])
print("score", metric["score"]["total"], "/", metric["score"]["max_possible"])


## Final public artifact

The bundled `solution/artifacts` is trained on all usable public sessions. It is intended for the hidden test. Never evaluate one of its training sessions as an independent held-out result. Rebuild it in a different output folder with the command below if needed.


In [ ]:
# Optional: about two minutes on the reference machine.
# subprocess.run([sys.executable, "train.py", "--data", str(DATA), "--out", "rebuilt_artifacts", "--cache-per-class", "400", "--stride-sec", "0.5", "--seed", "2026"], check=True)


## Public metrics without subject leakage

The competition also asks for metrics on the public training files. These commands rebuild 14 subject-excluding caches and evaluate all 56 files. They can take tens of minutes. The all-public final artifact is never used to score a public file in this protocol.


In [ ]:
# Optional full reproduction:

# subprocess.run([sys.executable, "build_public_loso.py", "--data", str(DATA), "--out", "results/metrics/fold_artifacts"], check=True)
# subprocess.run([sys.executable, "evaluate_public_loso.py", "--data", str(DATA), "--out", "results/metrics"], check=True)
